# Deep Sets Issue #120 — feature regimes

This notebook summarizes validation AUC when `metrics.json` is present under each experiment’s `train/` tree, lists point-feature columns for the registered configs, and documents how to launch the full build -> merge -> train pipeline on Slurm.

In [25]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def _resolve_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "build_deepsets_dataset.py").is_file():
            return p
    return here


REPO_ROOT = _resolve_repo_root()
_repo_root_str = str(REPO_ROOT)
if _repo_root_str not in sys.path:
    sys.path.insert(0, _repo_root_str)

deepsets_point_feature_names = importlib.import_module(
    "build_deepsets_dataset"
).deepsets_point_feature_names

FIG_DIR = REPO_ROOT / "analysis" / "figures" / "deepsets_issue120"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Registered Issue #120 point-feature configs. The first six are ISPY2-only;
# the final two run the same arms across all MAMA-MIA cohorts.
ARMS = [
    {
        "label": "ISPY2 baseline (curvature only)",
        "config": "configs/deepsets_ispy2_pointfeat_baseline.yaml",
        "regime": "baseline",
        "out_root": "experiments/deepsets_ispy2_pointfeat_baseline",
    },
    {
        "label": "ISPY2 geometry + topology",
        "config": "configs/deepsets_ispy2_pointfeat_geom_topo.yaml",
        "regime": "geometry_topology",
        "out_root": "experiments/deepsets_ispy2_pointfeat_geom_topo",
    },
    {
        "label": "ISPY2 geometry + topology + dynamic",
        "config": "configs/deepsets_ispy2_pointfeat_geom_topo_dynamic.yaml",
        "regime": "geometry_topology_dynamic",
        "out_root": "experiments/deepsets_ispy2_pointfeat_geom_topo_dynamic",
    },
    {
        "label": "ISPY2 geometry + topology + curvature",
        "config": "configs/deepsets_ispy2_pointfeat_geom_topo_plus_curvature.yaml",
        "regime": "geometry_topology_plus_curvature",
        "out_root": "experiments/deepsets_ispy2_pointfeat_geom_topo_plus_curvature",
    },
    {
        "label": "ISPY2 geometry + topology (no shells)",
        "config": "configs/deepsets_ispy2_pointfeat_geom_topo_no_shells.yaml",
        "regime": "geometry_topology_no_shells",
        "out_root": "experiments/deepsets_ispy2_pointfeat_geom_topo_no_shells",
    },
    {
        "label": "ISPY2 curvature + dynamic",
        "config": "configs/deepsets_ispy2_pointfeat_curvature_plus_dynamic.yaml",
        "regime": "curvature_plus_dynamic",
        "out_root": "experiments/deepsets_ispy2_pointfeat_curvature_plus_dynamic",
    },
    {
        "label": "All cohorts baseline (curvature only)",
        "config": "configs/deepsets_mamamia_pointfeat_baseline.yaml",
        "regime": "baseline",
        "cohort": "all_cohorts",
        "out_root": "experiments/deepsets_mamamia_pointfeat_baseline",
    },
    {
        "label": "All cohorts geometry + topology + curvature",
        "config": "configs/deepsets_mamamia_pointfeat_geom_topo_plus_curvature.yaml",
        "regime": "geometry_topology_plus_curvature",
        "cohort": "all_cohorts",
        "out_root": "experiments/deepsets_mamamia_pointfeat_geom_topo_plus_curvature",
    },
]


def find_latest_deepsets_metrics(out_root: Path) -> Path | None:
    """Newest metrics.json under <out_root>/train/ (any run timestamp / nested layout)."""
    train_root = out_root / "train"
    if not train_root.is_dir():
        return None
    candidates = list(train_root.rglob("metrics.json"))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def slurm_submit_one_liner(config_rel: str, out_root_rel: str) -> str:
    """Return the shell command used to submit one Deep Sets pipeline run."""
    return (
        f"CONFIG={config_rel} OUT_ROOT={out_root_rel} "
        f"BUILD_SHARDS=8 PARTITION=general "
        f"bash slurm/submit_deepsets_pipeline.sh"
    )

## Benchmark metrics (uses latest `metrics.json` per experiment if present)

In [26]:
rows = []
for arm in ARMS:
    out = REPO_ROOT / arm["out_root"]
    metrics_file = find_latest_deepsets_metrics(out)
    n_features = len(deepsets_point_feature_names(arm["regime"]))
    row = {
        "arm": arm["label"],
        "config": arm["config"],
        "regime": arm["regime"],
        "out_root": arm["out_root"],
        "n_features": n_features,
        "auc_mean": float("nan"),
        "auc_std": float("nan"),
        "metrics_path": "",
        "status": "missing",
    }
    if metrics_file is not None:
        payload = json.loads(metrics_file.read_text(encoding="utf-8"))
        row["auc_mean"] = float(payload["aggregated_metrics"]["auc"]["mean"])
        row["auc_std"] = float(payload["aggregated_metrics"]["auc"]["std"])
        row["metrics_path"] = str(metrics_file)
        row["status"] = "found"
    rows.append(row)

metrics_df = pd.DataFrame(rows)
found_df = metrics_df.loc[metrics_df["status"] == "found"]
missing_df = metrics_df.loc[metrics_df["status"] == "missing"]

print(
    f"Metrics loaded for {len(found_df)}/{len(metrics_df)} experiments "
    f"(newest metrics.json under each out_root/train/)."
)
if not missing_df.empty:
    print("\nNo metrics yet for:")
    for _, r in missing_df.iterrows():
        print(f"  - {r['arm']} ({r['out_root']})")
    print("\nRun the original ISPY2 six-arm array on Slurm (from repo root):")
    print("  sbatch slurm/submit_issue120_deepsets_feature_arms.slurm")
    print("\nOr run any missing pipeline locally / by hand:")
    for _, r in missing_df.iterrows():
        print(" ", slurm_submit_one_liner(r["config"], r["out_root"]))

metrics_df

Metrics loaded for 4/8 experiments (newest metrics.json under each out_root/train/).

No metrics yet for:
  - ISPY2 geometry + topology + dynamic (experiments/deepsets_ispy2_pointfeat_geom_topo_dynamic)
  - ISPY2 curvature + dynamic (experiments/deepsets_ispy2_pointfeat_curvature_plus_dynamic)
  - All cohorts baseline (curvature only) (experiments/deepsets_mamamia_pointfeat_baseline)
  - All cohorts geometry + topology + curvature (experiments/deepsets_mamamia_pointfeat_geom_topo_plus_curvature)

Run the original ISPY2 six-arm array on Slurm (from repo root):
  sbatch slurm/submit_issue120_deepsets_feature_arms.slurm

Or run any missing pipeline locally / by hand:
  CONFIG=configs/deepsets_ispy2_pointfeat_geom_topo_dynamic.yaml OUT_ROOT=experiments/deepsets_ispy2_pointfeat_geom_topo_dynamic BUILD_SHARDS=8 PARTITION=general bash slurm/submit_deepsets_pipeline.sh
  CONFIG=configs/deepsets_ispy2_pointfeat_curvature_plus_dynamic.yaml OUT_ROOT=experiments/deepsets_ispy2_pointfeat_curvature_

,arm,config,regime,out_root,n_features,auc_mean,auc_std,metrics_path,status
0,ISPY2 baseline (curvature only),configs/deepsets_ispy2_pointfeat_baseline.yaml,baseline,experiments/deepsets_ispy2_pointfeat_baseline,1,0.532344,0.029390,/home/lunad/vanguard-issue120-benchmark/experi...,found
1,ISPY2 geometry + topology,configs/deepsets_ispy2_pointfeat_geom_topo.yaml,geometry_topology,experiments/deepsets_ispy2_pointfeat_geom_topo,16,0.513844,0.009554,/home/lunad/vanguard-issue120-benchmark/experi...,found
2,ISPY2 geometry + topology + dynamic,configs/deepsets_ispy2_pointfeat_geom_topo_dyn...,geometry_topology_dynamic,experiments/deepsets_ispy2_pointfeat_geom_topo...,27,NaN,NaN,,missing
3,ISPY2 geometry + topology + curvature,configs/deepsets_ispy2_pointfeat_geom_topo_plu...,geometry_topology_plus_curvature,experiments/deepsets_ispy2_pointfeat_geom_topo...,17,0.534455,0.024683,/home/lunad/vanguard-issue120-benchmark/experi...,found
4,ISPY2 geometry + topology (no shells),configs/deepsets_ispy2_pointfeat_geom_topo_no_...,geometry_topology_no_shells,experiments/deepsets_ispy2_pointfeat_geom_topo...,12,0.529008,0.033948,/home/lunad/vanguard-issue120-benchmark/experi...,found
5,ISPY2 curvature + dynamic,configs/deepsets_ispy2_pointfeat_curvature_plu...,curvature_plus_dynamic,experiments/deepsets_ispy2_pointfeat_curvature...,12,NaN,NaN,,missing
6,All cohorts baseline (curvature only),configs/deepsets_mamamia_pointfeat_baseline.yaml,baseline,experiments/deepsets_mamamia_pointfeat_baseline,1,NaN,NaN,,missing
7,All cohorts geometry + topology + curvature,configs/deepsets_mamamia_pointfeat_geom_topo_p...,geometry_topology_plus_curvature,experiments/deepsets_mamamia_pointfeat_geom_to...,17,NaN,NaN,,missing


In [27]:
plot_df = metrics_df.loc[metrics_df["status"] == "found"]
if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.bar(
        plot_df["arm"],
        plot_df["auc_mean"],
        yerr=plot_df["auc_std"],
        capsize=6,
    )
    ax.set_ylabel("Validation AUC (mean ± std)")
    ax.set_ylim(0.45, 0.60)
    ax.set_title("Deep Sets feature-regime benchmark (runs with metrics present)")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=12, ha="right")
    fig.tight_layout()
    out_png = FIG_DIR / "feature_set_auc_comparison.png"
    fig.savefig(out_png, dpi=180)
    plt.close(fig)
    print(f"Saved: {out_png}")
else:
    print("No metrics found under any experiment train/ tree; skipped benchmark plot.")

Saved: /home/lunad/vanguard-issue120-benchmark/analysis/figures/deepsets_issue120/feature_set_auc_comparison.png


## Explicit feature lists by config

In [28]:
feature_rows = []
for arm in ARMS:
    names = deepsets_point_feature_names(arm["regime"])
    for i, feat in enumerate(names, start=1):
        feature_rows.append(
            {
                "config": arm["config"],
                "arm": arm["label"],
                "regime": arm["regime"],
                "feature_index": i,
                "feature_name": feat,
            }
        )

feature_table_df = pd.DataFrame(feature_rows)
feature_table_df

,config,arm,regime,feature_index,feature_name
0,configs/deepsets_ispy2_pointfeat_baseline.yaml,ISPY2 baseline (curvature only),baseline,1,curvature_rad
1,configs/deepsets_ispy2_pointfeat_geom_topo.yaml,ISPY2 geometry + topology,geometry_topology,1,signed_distance_mm
2,configs/deepsets_ispy2_pointfeat_geom_topo.yaml,ISPY2 geometry + topology,geometry_topology,2,abs_signed_distance_mm
3,configs/deepsets_ispy2_pointfeat_geom_topo.yaml,ISPY2 geometry + topology,geometry_topology,3,inside_tumor
4,configs/deepsets_ispy2_pointfeat_geom_topo.yaml,ISPY2 geometry + topology,geometry_topology,4,shell_0_2mm
...,...,...,...,...,...
98,configs/deepsets_mamamia_pointfeat_geom_topo_p...,All cohorts geometry + topology + curvature,geometry_topology_plus_curvature,13,offset_y_mm
99,configs/deepsets_mamamia_pointfeat_geom_topo_p...,All cohorts geometry + topology + curvature,geometry_topology_plus_curvature,14,offset_z_mm
100,configs/deepsets_mamamia_pointfeat_geom_topo_p...,All cohorts geometry + topology + curvature,geometry_topology_plus_curvature,15,support_radius_mm
101,configs/deepsets_mamamia_pointfeat_geom_topo_p...,All cohorts geometry + topology + curvature,geometry_topology_plus_curvature,16,support_radius_available


In [29]:
for arm in ARMS:
    display_df = feature_table_df.loc[
        feature_table_df["config"] == arm["config"], ["feature_index", "feature_name"]
    ]
    print(f"\n{arm['label']} :: {arm['config']} ({len(display_df)} features)")
    display(display_df.reset_index(drop=True))


ISPY2 baseline (curvature only) :: configs/deepsets_ispy2_pointfeat_baseline.yaml (1 features)


,feature_index,feature_name
0,1,curvature_rad



ISPY2 geometry + topology :: configs/deepsets_ispy2_pointfeat_geom_topo.yaml (16 features)


,feature_index,feature_name
0,1,signed_distance_mm
1,2,abs_signed_distance_mm
2,3,inside_tumor
3,4,shell_0_2mm
4,5,shell_2_5mm
5,6,shell_5_10mm
6,7,shell_ge_10mm
7,8,degree
8,9,is_endpoint
9,10,is_chain



ISPY2 geometry + topology + dynamic :: configs/deepsets_ispy2_pointfeat_geom_topo_dynamic.yaml (27 features)


...

,feature_index,feature_name
0,1,curvature_rad
1,2,arrival_index_norm
2,3,has_arrival
3,4,peak_index_norm
4,5,peak_enhancement
5,6,washin_slope
6,7,washout_slope
7,8,positive_enhancement_auc
8,9,peak_rel_reference
9,10,auc_rel_reference



All cohorts baseline (curvature only) :: configs/deepsets_mamamia_pointfeat_baseline.yaml (1 features)


,feature_index,feature_name
0,1,curvature_rad



All cohorts geometry + topology + curvature :: configs/deepsets_mamamia_pointfeat_geom_topo_plus_curvature.yaml (17 features)


,feature_index,feature_name
0,1,signed_distance_mm
1,2,abs_signed_distance_mm
2,3,inside_tumor
3,4,shell_0_2mm
4,5,shell_2_5mm
5,6,shell_5_10mm
6,7,shell_ge_10mm
7,8,degree
8,9,is_endpoint
9,10,is_chain


## Slurm: launch registered pipelines

In [30]:
launch_rows = []
for arm in ARMS:
    launch_rows.append(
        {
            "arm": arm["label"],
            "config": arm["config"],
            "out_root": arm["out_root"],
            "submit_deepsets_pipeline_command": slurm_submit_one_liner(
                arm["config"], arm["out_root"]
            ),
        }
    )

pd.DataFrame(launch_rows)

,arm,config,out_root,submit_deepsets_pipeline_command
0,ISPY2 baseline (curvature only),configs/deepsets_ispy2_pointfeat_baseline.yaml,experiments/deepsets_ispy2_pointfeat_baseline,CONFIG=configs/deepsets_ispy2_pointfeat_baseli...
1,ISPY2 geometry + topology,configs/deepsets_ispy2_pointfeat_geom_topo.yaml,experiments/deepsets_ispy2_pointfeat_geom_topo,CONFIG=configs/deepsets_ispy2_pointfeat_geom_t...
2,ISPY2 geometry + topology + dynamic,configs/deepsets_ispy2_pointfeat_geom_topo_dyn...,experiments/deepsets_ispy2_pointfeat_geom_topo...,CONFIG=configs/deepsets_ispy2_pointfeat_geom_t...
3,ISPY2 geometry + topology + curvature,configs/deepsets_ispy2_pointfeat_geom_topo_plu...,experiments/deepsets_ispy2_pointfeat_geom_topo...,CONFIG=configs/deepsets_ispy2_pointfeat_geom_t...
4,ISPY2 geometry + topology (no shells),configs/deepsets_ispy2_pointfeat_geom_topo_no_...,experiments/deepsets_ispy2_pointfeat_geom_topo...,CONFIG=configs/deepsets_ispy2_pointfeat_geom_t...
5,ISPY2 curvature + dynamic,configs/deepsets_ispy2_pointfeat_curvature_plu...,experiments/deepsets_ispy2_pointfeat_curvature...,CONFIG=configs/deepsets_ispy2_pointfeat_curvat...
6,All cohorts baseline (curvature only),configs/deepsets_mamamia_pointfeat_baseline.yaml,experiments/deepsets_mamamia_pointfeat_baseline,CONFIG=configs/deepsets_mamamia_pointfeat_base...
7,All cohorts geometry + topology + curvature,configs/deepsets_mamamia_pointfeat_geom_topo_p...,experiments/deepsets_mamamia_pointfeat_geom_to...,CONFIG=configs/deepsets_mamamia_pointfeat_geom...


The table above emits one `submit_deepsets_pipeline.sh` command per registered config, including the two all-cohort MAMA-MIA runs.

From the repository root (so paths and `SLURM_SUBMIT_DIR` resolve correctly), the existing array job still enqueues the original six ISPY2-only pipelines:

```bash
sbatch slurm/submit_issue120_deepsets_feature_arms.slurm
```

Sequential alternative for the original ISPY2 arms (also submits Slurm pipelines when `sbatch` is available): `bash scripts/benchmark_deepsets_feature_sets.sh`.

Override partition or shard count for the array: `PARTITION=gpu BUILD_SHARDS=16 sbatch slurm/submit_issue120_deepsets_feature_arms.slurm`.

In [31]:
feature_csv = FIG_DIR / "feature_columns_by_config.csv"
feature_table_df.to_csv(feature_csv, index=False)
print(f"Saved: {feature_csv}")

summary_csv = FIG_DIR / "feature_set_benchmark_summary.csv"
if metrics_df["status"].eq("found").any():
    metrics_df.to_csv(summary_csv, index=False)
    print(f"Saved: {summary_csv}")
else:
    print(
        f"Skipped {summary_csv.name} (no metrics yet). "
        "Re-run this cell after training completes."
    )

Saved: /home/lunad/vanguard-issue120-benchmark/analysis/figures/deepsets_issue120/feature_columns_by_config.csv
Saved: /home/lunad/vanguard-issue120-benchmark/analysis/figures/deepsets_issue120/feature_set_benchmark_summary.csv
